In [3]:
import pandas as pd

df = pd.read_csv(
    '../data/food.csv.gz',
    sep='\t',                  # TAB separated, not comma!
    compression='gzip',
    nrows=500000,              # Only load first 500k rows
    low_memory=False
)

print(df.shape)
print(df.columns.tolist())

(500000, 210)
['code', 'url', 'creator', 'created_t', 'created_datetime', 'last_modified_t', 'last_modified_datetime', 'last_modified_by', 'last_updated_t', 'last_updated_datetime', 'product_name', 'abbreviated_product_name', 'generic_name', 'quantity', 'packaging', 'packaging_tags', 'packaging_en', 'packaging_text', 'brands', 'brands_tags', 'brands_en', 'categories', 'categories_tags', 'categories_en', 'origins', 'origins_tags', 'origins_en', 'manufacturing_places', 'manufacturing_places_tags', 'labels', 'labels_tags', 'labels_en', 'emb_codes', 'emb_codes_tags', 'first_packaging_code_geo', 'cities', 'cities_tags', 'purchase_places', 'stores', 'countries', 'countries_tags', 'countries_en', 'ingredients_text', 'ingredients_tags', 'ingredients_analysis_tags', 'allergens', 'allergens_en', 'traces', 'traces_tags', 'traces_en', 'serving_size', 'serving_quantity', 'no_nutrition_data', 'additives_n', 'additives', 'additives_tags', 'additives_en', 'nutriscore_score', 'nutriscore_grade', 'nova_

In [4]:
# Find our key columns
sugar_cols = [col for col in df.columns if 'sugar' in col.lower()]
protein_cols = [col for col in df.columns if 'protein' in col.lower()]
fat_cols = [col for col in df.columns if 'fat' in col.lower()]
fiber_cols = [col for col in df.columns if 'fiber' in col.lower()]
category_cols = [col for col in df.columns if 'categor' in col.lower()]

print("Sugar columns:", sugar_cols)
print("Protein columns:", protein_cols)
print("Fat columns:", fat_cols)
print("Fiber columns:", fiber_cols)
print("Category columns:", category_cols)

Sugar columns: ['sugars_100g', 'added-sugars_100g']
Protein columns: ['proteins_100g', 'serum-proteins_100g', 'collagen-meat-protein-ratio_100g']
Fat columns: ['energy-from-fat_100g', 'fat_100g', 'saturated-fat_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-6-fat_100g', 'trans-fat_100g']
Fiber columns: ['fiber_100g', 'soluble-fiber_100g', 'insoluble-fiber_100g']
Category columns: ['categories', 'categories_tags', 'categories_en', 'main_category', 'main_category_en']


In [5]:
# Preview the key columns we expect to use
key_cols = ['product_name', 'categories_tags', 'ingredients_text']
print(df[key_cols].head(3))

                    product_name categories_tags  \
0  Limonade artisanale a la rose             NaN   
1                  M&amp;M white             NaN   
2                   Chocolate n3             NaN   

                                    ingredients_text  
0                                                NaN  
1  Weizenmehl, Rapsöl, Speisesalz, 1,7% Meersalz,...  
2                                                NaN  


In [6]:
# Check how much data is missing in our key columns
key_cols = ['product_name', 'categories_tags', 'ingredients_text', 
            'sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g']

missing = df[key_cols].isnull().sum()
total = len(df)

print("Missing values:\n")
for col, count in missing.items():
    pct = (count / total) * 100
    print(f"  {col}: {count} missing ({pct:.1f}%)")

Missing values:

  product_name: 15690 missing (3.1%)
  categories_tags: 231471 missing (46.3%)
  ingredients_text: 232286 missing (46.5%)
  sugars_100g: 394381 missing (78.9%)
  proteins_100g: 391135 missing (78.2%)
  fat_100g: 391276 missing (78.3%)
  fiber_100g: 422247 missing (84.4%)


In [7]:
# ============================================
# STORY 1: DATA CLEANING
# ============================================

print("Shape before cleaning:", df.shape)

# Step 1: Keep only the columns we actually need
cols_to_keep = [
    'product_name',
    'categories_tags',
    'ingredients_text',
    'sugars_100g',
    'proteins_100g',
    'fat_100g',
    'fiber_100g'
]
df_clean = df[cols_to_keep].copy()

# Step 2: Drop rows missing product_name, sugars, or proteins
# (these are essential for our analysis)
df_clean = df_clean.dropna(subset=['product_name', 'sugars_100g', 'proteins_100g'])

print("Shape after dropping missing essentials:", df_clean.shape)

# Step 3: Remove biologically impossible values
# (per 100g, nothing can exceed 100g)
df_clean = df_clean[
    (df_clean['sugars_100g'] >= 0) & (df_clean['sugars_100g'] <= 100) &
    (df_clean['proteins_100g'] >= 0) & (df_clean['proteins_100g'] <= 100) &
    (df_clean['fat_100g'].isna() | ((df_clean['fat_100g'] >= 0) & (df_clean['fat_100g'] <= 100)))
]

print("Shape after removing impossible values:", df_clean.shape)

# Step 4: Reset index
df_clean = df_clean.reset_index(drop=True)

print("\n✅ Cleaning complete!")
print("Final clean dataset shape:", df_clean.shape)
print("\nMissing values in clean dataset:")
print(df_clean.isnull().sum())

Shape before cleaning: (500000, 210)
Shape after dropping missing essentials: (103859, 7)
Shape after removing impossible values: (103682, 7)

✅ Cleaning complete!
Final clean dataset shape: (103682, 7)

Missing values in clean dataset:
product_name            0
categories_tags     53632
ingredients_text    58882
sugars_100g             0
proteins_100g           0
fat_100g              207
fiber_100g          28169
dtype: int64


In [8]:
print("Shape before cleaning:", df.shape)

cols_to_keep = [
    'product_name',
    'categories_tags',
    'ingredients_text',
    'sugars_100g',
    'proteins_100g',
    'fat_100g',
    'fiber_100g'
]
df_clean = df[cols_to_keep].copy()

df_clean = df_clean.dropna(subset=['product_name', 'sugars_100g', 'proteins_100g'])

print("Shape after dropping missing essentials:", df_clean.shape)

df_clean = df_clean[
    (df_clean['sugars_100g'] >= 0) & (df_clean['sugars_100g'] <= 100) &
    (df_clean['proteins_100g'] >= 0) & (df_clean['proteins_100g'] <= 100) &
    (df_clean['fat_100g'].isna() | ((df_clean['fat_100g'] >= 0) & (df_clean['fat_100g'] <= 100)))
]

print("Shape after removing impossible values:", df_clean.shape)

df_clean = df_clean.reset_index(drop=True)

print("\n✅ Cleaning complete!")
print("Final clean dataset shape:", df_clean.shape)
print("\nMissing values in clean dataset:")
print(df_clean.isnull().sum())

Shape before cleaning: (500000, 210)
Shape after dropping missing essentials: (103859, 7)
Shape after removing impossible values: (103682, 7)

✅ Cleaning complete!
Final clean dataset shape: (103682, 7)

Missing values in clean dataset:
product_name            0
categories_tags     53632
ingredients_text    58882
sugars_100g             0
proteins_100g           0
fat_100g              207
fiber_100g          28169
dtype: int64


In [9]:
# Preview categories_tags to understand what we're working with
print(df_clean['categories_tags'].dropna().head(10).tolist())

['en:asian-style-ready-meal', 'en:beverages-and-beverages-preparations,en:beverages', 'en:plant-based-foods-and-beverages,en:plant-based-foods,en:condiments,en:spices,en:curry-powder,en:powder,en:sauce-powder', 'en:snacks,en:sweet-snacks,en:biscuits-and-cakes,en:cakes,en:doughnuts', 'en:breakfasts,en:spreads,en:sweet-spreads,fr:pates-a-tartiner,en:hazelnut-spreads,en:chocolate-spreads,en:cocoa-and-hazelnuts-spreads', 'en:plant-based-foods-and-beverages,en:plant-based-foods,en:meat-alternatives,en:meat-analogues,en:vegetarian-patties,en:vegan-patties', 'en:meats-and-their-products,en:meats,en:prepared-meats,en:cured-sausages,en:chorizo', 'en:meats-and-their-products,en:meats,en:prepared-meats,en:cured-sausages,en:chorizo', 'en:seafood,en:fishes-and-their-products,en:fishes,en:lean-fishes,en:mullet', 'en:beverages,en:carbonated-drinks,en:sodas,en:colas']


In [11]:
# ============================================
# STORY 2: IMPROVED CATEGORY GROUPING
# ============================================

def assign_category(tags):
    if not isinstance(tags, str):
        return 'Other'
    
    tags = tags.lower()
    
    # Order matters — more specific first!
    if any(k in tags for k in ['chocolate', 'candy', 'confectionery', 'sweet-snack', 'sugar-confectionery']):
        return 'Confectionery'
    elif any(k in tags for k in ['biscuit', 'cookie', 'cake', 'pastry', 'doughnut', 'wafer', 'muffin', 'brownie']):
        return 'Biscuits & Cakes'
    elif any(k in tags for k in ['snack', 'chip', 'crisp', 'popcorn', 'pretzel', 'cracker', 'puff']):
        return 'Snacks'
    elif any(k in tags for k in ['cereal', 'breakfast', 'granola', 'muesli', 'oat', 'porridge']):
        return 'Cereals & Breakfast'
    elif any(k in tags for k in ['bread', 'loaf', 'toast', 'bagel', 'roll', 'wrap', 'bakery', 'baked']):
        return 'Bread & Bakery'
    elif any(k in tags for k in ['spread', 'jam', 'honey', 'hazelnut', 'peanut-butter', 'marmalade']):
        return 'Spreads'
    elif any(k in tags for k in ['yogurt', 'yoghurt', 'cheese', 'milk', 'dairy', 'butter', 'cream']):
        return 'Dairy'
    elif any(k in tags for k in ['meat', 'chicken', 'beef', 'pork', 'sausage', 'chorizo', 'ham', 'bacon', 'poultry']):
        return 'Meat & Poultry'
    elif any(k in tags for k in ['seafood', 'fish', 'tuna', 'salmon', 'shrimp', 'prawn']):
        return 'Seafood'
    elif any(k in tags for k in ['beverage', 'drink', 'juice', 'soda', 'water', 'cola', 'tea', 'coffee', 'smoothie', 'milk-based-beverage']):
        return 'Beverages'
    elif any(k in tags for k in ['sauce', 'condiment', 'spice', 'seasoning', 'dressing', 'vinegar', 'ketchup', 'mustard']):
        return 'Sauces & Condiments'
    elif any(k in tags for k in ['vegetable', 'fruit', 'plant-based', 'vegan', 'legume', 'bean', 'lentil', 'tofu', 'nuts']):
        return 'Plant Based'
    elif any(k in tags for k in ['pasta', 'rice', 'noodle', 'grain', 'flour', 'starch']):
        return 'Grains & Pasta'
    elif any(k in tags for k in ['frozen', 'ready-meal', 'prepared', 'instant']):
        return 'Ready Meals'
    else:
        return 'Other'

# Apply improved function
df_clean['primary_category'] = df_clean['categories_tags'].apply(assign_category)

# Check the distribution
print("Category Distribution:")
print(df_clean['primary_category'].value_counts())
print(f"\nTotal categories: {df_clean['primary_category'].nunique()}")
print(f"\nOther percentage: {(df_clean['primary_category'] == 'Other').sum() / len(df_clean) * 100:.1f}%")

Category Distribution:
primary_category
Other                  57800
Cereals & Breakfast    11891
Beverages               7962
Dairy                   7350
Confectionery           6083
Snacks                  2841
Sauces & Condiments     2795
Meat & Poultry          2781
Spreads                 1097
Biscuits & Cakes         770
Ready Meals              760
Seafood                  734
Bread & Bakery           372
Grains & Pasta           292
Plant Based              154
Name: count, dtype: int64

Total categories: 15

Other percentage: 55.7%


In [12]:
# Let's peek inside the 'Other' bucket
other_sample = df_clean[df_clean['primary_category'] == 'Other']['categories_tags'].dropna().head(20).tolist()

for item in other_sample:
    print(item)
    print("---")

en:dietary-supplements,en:bodybuilding-supplements,en:protein-powders
---
en:dietary-supplements,en:bodybuilding-supplements,en:protein-shakes
---
en:meals,en:soups,en:cold-soups
---
en:meals,en:soups,en:reheatable-soups
---
en:meals,en:soups
---
en:meals,en:soups,en:cold-soups
---
fr:potage
---
en:meals,en:soups,en:reheatable-soups
---
en:food
---
en:dietary-supplements
---
en:dietary-supplements,en:vitamin-mineral-combinations
---
en:meals,en:dietary-supplements,en:meal-replacements
---
en:dietary-supplements
---
en:dietary-supplements,en:bodybuilding-supplements,en:protein-powders
---
en:sweet-pies,en:pies,en:coconut-pies
---
en:bonbon-pastille
---
en:dietary-supplements,en:bodybuilding-supplements,en:protein-powders
---
en:meals,en:dietary-supplements,en:bodybuilding-supplements,en:protein-powders
---
en:dietary-supplements,en:bodybuilding-supplements
---
en:dietary-supplements,en:bodybuilding-supplements
---


In [13]:
def assign_category(tags):
    if not isinstance(tags, str):
        return 'Other'
    
    tags = tags.lower()
    
    if any(k in tags for k in ['protein-powder', 'protein-shake', 'bodybuilding', 'dietary-supplement', 'meal-replacement', 'vitamin', 'mineral']):
        return 'Supplements & Protein'
    elif any(k in tags for k in ['soup', 'broth', 'bouillon', 'potage']):
        return 'Soups'
    elif any(k in tags for k in ['chocolate', 'candy', 'confectionery', 'sweet-snack', 'sugar-confectionery']):
        return 'Confectionery'
    elif any(k in tags for k in ['biscuit', 'cookie', 'cake', 'pastry', 'doughnut', 'wafer', 'muffin', 'brownie']):
        return 'Biscuits & Cakes'
    elif any(k in tags for k in ['snack', 'chip', 'crisp', 'popcorn', 'pretzel', 'cracker', 'puff']):
        return 'Snacks'
    elif any(k in tags for k in ['cereal', 'breakfast', 'granola', 'muesli', 'oat', 'porridge']):
        return 'Cereals & Breakfast'
    elif any(k in tags for k in ['bread', 'loaf', 'toast', 'bagel', 'roll', 'wrap', 'bakery', 'baked']):
        return 'Bread & Bakery'
    elif any(k in tags for k in ['spread', 'jam', 'honey', 'hazelnut', 'peanut-butter', 'marmalade']):
        return 'Spreads'
    elif any(k in tags for k in ['yogurt', 'yoghurt', 'cheese', 'milk', 'dairy', 'butter', 'cream']):
        return 'Dairy'
    elif any(k in tags for k in ['meat', 'chicken', 'beef', 'pork', 'sausage', 'chorizo', 'ham', 'bacon', 'poultry']):
        return 'Meat & Poultry'
    elif any(k in tags for k in ['seafood', 'fish', 'tuna', 'salmon', 'shrimp', 'prawn']):
        return 'Seafood'
    elif any(k in tags for k in ['beverage', 'drink', 'juice', 'soda', 'water', 'cola', 'tea', 'coffee', 'smoothie']):
        return 'Beverages'
    elif any(k in tags for k in ['sauce', 'condiment', 'spice', 'seasoning', 'dressing', 'vinegar', 'ketchup', 'mustard']):
        return 'Sauces & Condiments'
    elif any(k in tags for k in ['vegetable', 'fruit', 'plant-based', 'vegan', 'legume', 'bean', 'lentil', 'tofu', 'nuts']):
        return 'Plant Based'
    elif any(k in tags for k in ['pasta', 'rice', 'noodle', 'grain', 'flour', 'starch']):
        return 'Grains & Pasta'
    elif any(k in tags for k in ['frozen', 'ready-meal', 'prepared', 'instant']):
        return 'Ready Meals'
    else:
        return 'Other'

# Apply final version
df_clean['primary_category'] = df_clean['categories_tags'].apply(assign_category)

# Results
print("Category Distribution:")
print(df_clean['primary_category'].value_counts())
print(f"\nTotal categories: {df_clean['primary_category'].nunique()}")
print(f"\nOther percentage: {(df_clean['primary_category'] == 'Other').sum() / len(df_clean) * 100:.1f}%")

Category Distribution:
primary_category
Other                    56985
Cereals & Breakfast      11824
Beverages                 7700
Dairy                     7292
Confectionery             6022
Snacks                    2827
Sauces & Condiments       2781
Meat & Poultry            2717
Spreads                   1095
Biscuits & Cakes           769
Soups                      769
Ready Meals                754
Seafood                    731
Supplements & Protein      601
Bread & Bakery             370
Grains & Pasta             292
Plant Based                153
Name: count, dtype: int64

Total categories: 17

Other percentage: 55.0%


In [14]:
# How many 'Other' products have NO categories_tags at all?
other_df = df_clean[df_clean['primary_category'] == 'Other']

no_tags = other_df['categories_tags'].isna().sum()
has_tags = other_df['categories_tags'].notna().sum()

print(f"'Other' products with NO tags: {no_tags} ({no_tags/len(other_df)*100:.1f}%)")
print(f"'Other' products WITH tags: {has_tags} ({has_tags/len(other_df)*100:.1f}%)")

'Other' products with NO tags: 53632 (94.1%)
'Other' products WITH tags: 3353 (5.9%)


In [15]:
# Drop 'Other' category - they have no useful category information
df_final = df_clean[df_clean['primary_category'] != 'Other'].copy()
df_final = df_final.reset_index(drop=True)

print("Final dataset shape:", df_final.shape)
print(f"\nCategory Distribution:")
print(df_final['primary_category'].value_counts())
print(f"\nTotal products for analysis: {len(df_final)}")

Final dataset shape: (46697, 8)

Category Distribution:
primary_category
Cereals & Breakfast      11824
Beverages                 7700
Dairy                     7292
Confectionery             6022
Snacks                    2827
Sauces & Condiments       2781
Meat & Poultry            2717
Spreads                   1095
Biscuits & Cakes           769
Soups                      769
Ready Meals                754
Seafood                    731
Supplements & Protein      601
Bread & Bakery             370
Grains & Pasta             292
Plant Based                153
Name: count, dtype: int64

Total products for analysis: 46697
